In [1]:
import pandas as pd

In [2]:
# Use the full absolute path
df = pd.read_csv(r"D:\bmw_intern\data\Parts.csv", delimiter=";")



In [3]:
df.head(2)

,ID,DESCRIPTION,Attribut1,Additional Feature,Application,Characteristic,Temp,Height,Length in mm,Rating,...,Operating Temperature-Min (Cel),Physical Dimension,Pre-arcing time-Min (ms),Product Diameter,Product Length,Rated Breaking Capacity (A),Rated Current (A),Rated Voltage (V),Rated Voltage(AC) (V),Rated Voltage(DC) (V)
0,A1,Indicator Red Fast Movement 1.6A 250V Holder P...,Fast,NaN,Primary Protection In Equipment,VERY FAST,NaN,20mm,5.2mm,1.6A,...,-55Cel,5.2mm x 20mm,3ms,5.2mm,20mm,1500A,1.6A,250V,250V,NaN
1,A2,"Non Resettable Indicators Electric Indicator, ...",NaN,NaN,Primary Protection In Equipment,VERY FAST,NaN,20mm,5.2mm,NaN,...,-55Cel,5.2mm x 20mm,3ms,5.2mm,20mm,1500A,6.3A,250V,250V,NaN


In [4]:
df.columns.to_list()

['ID',
 'DESCRIPTION',
 'Attribut1',
 'Additional Feature',
 'Application',
 'Characteristic',
 'Temp',
 'Height',
 'Length in mm',
 'Rating',
 'Material',
 'Size',
 'Code',
 'Joule-integral-Nom (J)',
 'LC Risk',
 'Maximum AC Voltage Rating',
 'Maximum DC Voltage Rating',
 'Maximum Power Dissipation',
 'Mounting',
 'Mounting Feature',
 'Number of Terminals',
 'Operating Temperature-Max (Cel)',
 'Operating Temperature-Min (Cel)',
 'Physical Dimension',
 'Pre-arcing time-Min (ms)',
 'Product Diameter',
 'Product Length',
 'Rated Breaking Capacity (A)',
 'Rated Current (A)',
 'Rated Voltage (V)',
 'Rated Voltage(AC) (V)',
 'Rated Voltage(DC) (V)']

In [4]:
import pandas as pd

def find_alternative_parts_balanced(selected_part, df, min_alternatives=5):
    """
    Find alternative parts using a tiered filtering approach while ensuring at least `min_alternatives` results.
    First, strict filtering applies, prioritizing fuses based on the selected part's type (Fast Blow, Slow Blow, HRC, PTC, etc.).
    If strict filtering does not yield 5 alternatives, constraints are relaxed step-by-step.
    """
    # Extract key attributes of the selected part
    selected_current = selected_part["Rated Current (A)"]
    selected_voltage = selected_part["Rated Voltage (V)"]
    selected_breaking_capacity = selected_part["Rated Breaking Capacity (A)"]
    selected_mounting = selected_part["Mounting"]
    selected_application = selected_part["Application"]
    selected_fuse_type = selected_part["Attribut1"].strip().lower() if pd.notna(selected_part["Attribut1"]) else ""

    # Determine fuse type dynamically from dataset
    fuse_types = df["Attribut1"].dropna().str.lower().unique()
    fuse_filter = next((f for f in fuse_types if f in selected_fuse_type), "")

    # Tier 1: Strict Filtering (Match Fuse Type + Exact Electrical & Mechanical Match)
    strict_alternatives = df[
        (df["Rated Current (A)"] == selected_current) &
        (df["Rated Voltage (V)"] == selected_voltage) &
        (df["Rated Breaking Capacity (A)"] == selected_breaking_capacity) &
        (df["Mounting"] == selected_mounting) &
        (df["Attribut1"].str.lower().str.contains(fuse_filter, na=False) if fuse_filter else True)  # Match Fuse Type
    ]
    strict_alternatives = strict_alternatives[strict_alternatives["ID"] != selected_part["ID"]]  # Exclude itself

    if len(strict_alternatives) >= min_alternatives:
        print(f"Best Alternatives (Strict Filtering, {fuse_filter.capitalize()} Fuse Priority):")
        print(strict_alternatives.head(min_alternatives))
        return strict_alternatives.head(min_alternatives)

    # Tier 2: Relax Mechanical Constraints (Allow Similar Mounting/Size, Maintain Fuse Type Priority)
    relaxed_alternatives = df[
        (df["Rated Current (A)"] == selected_current) &
        (df["Rated Voltage (V)"] == selected_voltage) &
        (df["Rated Breaking Capacity (A)"] == selected_breaking_capacity) &
        (df["Attribut1"].str.lower().str.contains(fuse_filter, na=False) if fuse_filter else True)
    ]
    relaxed_alternatives = relaxed_alternatives[relaxed_alternatives["ID"] != selected_part["ID"]]

    if len(relaxed_alternatives) >= min_alternatives:
        print(f"Relaxed Alternatives ({fuse_filter.capitalize()} Fuse Priority, Verify Compatibility):")
        print(relaxed_alternatives.head(min_alternatives))
        return relaxed_alternatives.head(min_alternatives)

    # Tier 3: Allow Small Variations in Voltage & Breaking Capacity (±10% tolerance, Keep Fuse Type Priority)
    voltage_tolerance = 0.1  # Allow ±10% variation in voltage
    voltage_numeric = float(str(selected_voltage).replace("V", ""))
    breaking_capacity_numeric = float(str(selected_breaking_capacity).replace("A", "")) if pd.notna(selected_breaking_capacity) else None

    tolerated_alternatives = df[
        (df["Rated Current (A)"] == selected_current) &
        (df["Rated Voltage (V)"].str.replace("V", "").astype(float).between(
            voltage_numeric * (1 - voltage_tolerance), voltage_numeric * (1 + voltage_tolerance), inclusive="both"
        ))
    ]
    tolerated_alternatives = tolerated_alternatives[tolerated_alternatives["ID"] != selected_part["ID"]]

    if breaking_capacity_numeric:
        tolerated_alternatives = tolerated_alternatives[
            tolerated_alternatives["Rated Breaking Capacity (A)"].str.replace("A", "").astype(float).between(
                breaking_capacity_numeric * 0.9, breaking_capacity_numeric * 1.1, inclusive="both"
            )
        ]

    if len(tolerated_alternatives) >= min_alternatives:
        print("Further Relaxed Alternatives (Verify Carefully):")
        print(tolerated_alternatives.head(min_alternatives))
        return tolerated_alternatives.head(min_alternatives)

    # Tier 4: Functionally Similar Parts (Last Resort - Same Application Category)
    application_alternatives = df[df["Application"] == selected_application]
    application_alternatives = application_alternatives[application_alternatives["ID"] != selected_part["ID"]]

    print("Last Resort Alternatives (Same Application, Requires Review):")
    print(application_alternatives.head(min_alternatives))
    return application_alternatives.head(min_alternatives)



# Select a specific part by ID (A89 in this case)
sample_part = df[df["ID"] == "A89"].iloc[0]

# Find the best alternatives using the balanced logic
sample_alternatives = find_alternative_parts_balanced(sample_part, df, min_alternatives=5)

# Display the selected part and its alternatives
print(f"Selected Part: {sample_part['ID']} - {sample_part['DESCRIPTION']}")
print("Final Top 5 Alternatives:")
print(sample_alternatives)


Last Resort Alternatives (Same Application, Requires Review):
     ID                                        DESCRIPTION  Attribut1  \
21  A22  Indicator Red Slow Blow Movement 6.3A 250V Hol...  Slow Blow   
22  A23  Indicator Red Slow Blow Movement 16A 250V Hold...  Slow Blow   
23  A24  Indicator Red Slow Blow Movement 1A 250V Holde...  Slow Blow   
25  A26  Indicator Red Slow Blow Movement 1A 250V Holde...  Slow Blow   
26  A27  Indicator Red Slow Blow Movement 1.25A 250V Ho...  Slow Blow   

                            Additional Feature    Application Characteristic  \
21                                         NaN  Motor Circuit       TIME LAG   
22                                         NaN  Motor Circuit       TIME LAG   
23  RATED BREAKING CAPACITY AT 300 VDC: 1500 A  Motor Circuit       TIME LAG   
25  RATED BREAKING CAPACITY AT 300 VDC: 1500 A  Motor Circuit       TIME LAG   
26  RATED BREAKING CAPACITY AT 300 VDC: 1500 A  Motor Circuit       TIME LAG   

   Temp  Height Le

In [6]:
! pip install ipywidgets

   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ---------------------------------------- 2.3/2.3 MB 26.6 MB/s eta 0:00:00


DEPRECATION: Loading egg at c:\users\akshayanivashini\appdata\local\programs\python\python311\lib\site-packages\amllib-1.1-py3.11.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at c:\users\akshayanivashini\appdata\local\programs\python\python311\lib\site-packages\contourpy-1.2.0-py3.11-win-amd64.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at c:\users\akshayanivashini\appdata\local\programs\python\python311\lib\site-packages\cycler-0.12.1-py3.11.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECA

In [ ]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

def find_alternative_parts_balanced(selected_part, df, min_alternatives=5, relaxation_tier="Tier 1"):
    """
    Find alternative parts using a tiered filtering approach while ensuring at least `min_alternatives` results.
    Prioritizes fuse type, electrical characteristics, and mounting style before relaxing constraints.
    """

    # Extract key attributes of the selected part
    selected_current = float(str(selected_part["Rated Current (A)"]).replace("A", "").strip())
    selected_voltage_ac = selected_part["Rated Voltage (V)"]  # AC voltage
    selected_voltage_dc = selected_part["Rated Voltage (VDC)"] if "Rated Voltage (VDC)" in selected_part else None
    selected_breaking_capacity = selected_part["Rated Breaking Capacity (A)"]
    selected_mounting = selected_part["Mounting"]
    selected_application = selected_part["Application"]
    selected_fuse_type = selected_part["Attribut1"].strip().lower() if pd.notna(selected_part["Attribut1"]) else None

    print(f"Selected Part: {selected_part['ID']}")
    print(f"Fuse Type: {selected_fuse_type}")
    print(f"Rated Current: {selected_current}A")
    print(f"Rated Voltage (AC): {selected_voltage_ac}V")
    print(f"Rated Voltage (DC): {selected_voltage_dc}V" if selected_voltage_dc else "Rated Voltage (DC): N/A")
    print(f"Breaking Capacity: {selected_breaking_capacity}A")
    print(f"Mounting: {selected_mounting}")
    print(f"Application: {selected_application}")
    print(f"Relaxation Tier: {relaxation_tier}")

    # Remove rows with missing fuse types
    df = df.dropna(subset=["Attribut1"])

    # Convert numerical columns safely
    df["Rated Current (A)"] = pd.to_numeric(df["Rated Current (A)"].astype(str).str.replace("A", ""), errors='coerce')
    df["Rated Voltage (V)"] = pd.to_numeric(df["Rated Voltage (V)"].astype(str).str.replace("V", ""), errors='coerce')
    if "Rated Voltage (VDC)" in df.columns:
        df["Rated Voltage (VDC)"] = pd.to_numeric(df["Rated Voltage (VDC)"].astype(str).str.replace("V", ""), errors='coerce')
    
    df["Rated Breaking Capacity (A)"] = pd.to_numeric(df["Rated Breaking Capacity (A)"].astype(str).str.replace("A", ""), errors='coerce')

    # Fill missing numerical values with the median
    df["Rated Voltage (V)"].fillna(df["Rated Voltage (V)"].median(), inplace=True)
    if "Rated Voltage (VDC)" in df.columns:
        df["Rated Voltage (VDC)"].fillna(df["Rated Voltage (VDC)"].median(), inplace=True)

    df["Rated Breaking Capacity (A)"].fillna(df["Rated Breaking Capacity (A)"].median(), inplace=True)

    # Convert selected values to float safely
    selected_voltage_ac = pd.to_numeric(str(selected_voltage_ac).replace("V", ""), errors='coerce')
    selected_voltage_dc = pd.to_numeric(str(selected_voltage_dc).replace("V", ""), errors='coerce') if selected_voltage_dc else None
    selected_breaking_capacity = pd.to_numeric(str(selected_breaking_capacity).replace("A", ""), errors='coerce')

    # Use median if selected values are missing
    if pd.isna(selected_voltage_ac):
        selected_voltage_ac = df["Rated Voltage (V)"].median()
    
    if selected_voltage_dc is not None and pd.isna(selected_voltage_dc):
        selected_voltage_dc = df["Rated Voltage (VDC)"].median()

    if pd.isna(selected_breaking_capacity):
        selected_breaking_capacity = df["Rated Breaking Capacity (A)"].median()

    # Ensure voltage filtering is correct for both AC and DC
    df = df[df["Rated Voltage (V)"] >= selected_voltage_ac]  # Ensure only equal or higher AC voltage
    if "Rated Voltage (VDC)" in df.columns and selected_voltage_dc is not None:
        df = df[df["Rated Voltage (VDC)"] >= selected_voltage_dc]  # Ensure only equal or higher DC voltage

    # Ensure only exact fuse type matches
    df = df[df["Attribut1"].str.lower() == selected_fuse_type]

    # 🛠 **Tier 1: Strict Matching (Fuse Type + Exact Electrical & Mechanical Match)**
    if relaxation_tier == "Tier 1":
        alternatives = df[
            (df["Rated Current (A)"].between(selected_current - 2, selected_current + 2)) &  # Allow ±2A range
            (df["Rated Breaking Capacity (A)"].between(selected_breaking_capacity * 0.9, selected_breaking_capacity * 1.1)) &
            (df["Mounting"] == selected_mounting)
        ]
    # 🛠 **Tier 2: Allow Minor Breaking Capacity Variations**
    elif relaxation_tier == "Tier 2":
        alternatives = df[
            (df["Rated Current (A)"].between(selected_current - 2, selected_current + 2)) &
            (df["Rated Breaking Capacity (A)"].between(selected_breaking_capacity * 0.8, selected_breaking_capacity * 1.2))
        ]
    # 🛠 **Tier 3: Relax Current Tolerance Further (±5A)**
    elif relaxation_tier == "Tier 3":
        alternatives = df[
            (df["Rated Current (A)"].between(selected_current - 5, selected_current + 5)) &
            (df["Rated Breaking Capacity (A)"].between(selected_breaking_capacity * 0.8, selected_breaking_capacity * 1.2))
        ]
    # 🛠 **Tier 4: Allow Different Mounting Styles**
    elif relaxation_tier == "Tier 4":
        alternatives = df[
            (df["Rated Current (A)"].between(selected_current - 5, selected_current + 5)) &
            (df["Rated Breaking Capacity (A)"].between(selected_breaking_capacity * 0.8, selected_breaking_capacity * 1.2))
        ]
    # 🛠 **Tier 5: Allow Different Fuse Types**
    elif relaxation_tier == "Tier 5":
        alternatives = df[
            (df["Rated Current (A)"].between(selected_current - 5, selected_current + 5)) &
            (df["Rated Breaking Capacity (A)"].between(selected_breaking_capacity * 0.8, selected_breaking_capacity * 1.2))
        ]
    else:
        alternatives = pd.DataFrame()  # Default to empty DataFrame

    alternatives = alternatives[alternatives["ID"] != selected_part["ID"]]
    print(f"Found {len(alternatives)} alternatives.")
    return alternatives.head(min_alternatives)


selected_part = df[df["ID"] == "A89"].iloc[0]

# Interactive Buttons for Relaxation Tiers
def on_button_click(tier):
    print(f"\nApplying {tier}...")
    alternatives = find_alternative_parts_balanced(selected_part, df, relaxation_tier=tier)
    print("Alternatives:")
    print(alternatives)

# Create buttons for each tier
tier_buttons = [
    widgets.Button(description="Tier 1: Strict Matching"),
    widgets.Button(description="Tier 2: Relax Breaking Capacity"),
    widgets.Button(description="Tier 3: Relax Current Tolerance"),
    widgets.Button(description="Tier 4: Allow Different Mounting"),
    widgets.Button(description="Tier 5: Allow Different Fuse Types"),
]

# Assign button click handlers
for i, button in enumerate(tier_buttons):
    button.on_click(lambda b, tier=f"Tier {i + 1}": on_button_click(tier))

# Display buttons
display(widgets.VBox(tier_buttons))